## Assignment 5

In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.ensemble import RandomForestClassifier

sns.set(style="whitegrid")
pd.set_option("display.max_columns", None)

DATA_PATH = "C:\\Users\\joury\\Downloads\\talabat_enhanced_orders2.csv"
df = pd.read_csv(DATA_PATH)

if "Item_Name" in df.columns:
    top_k = 6
    top_items = df["Item_Name"].value_counts().head(top_k).index
    df["Item_Name_reduced"] = np.where(
        df["Item_Name"].isin(top_items),
        df["Item_Name"],
        "Other"
    )

    print("Unique Item_Name:", df["Item_Name"].nunique())
    print("Unique Item_Name_reduced:", df["Item_Name_reduced"].nunique())
    print(df[["Item_Name", "Item_Name_reduced"]].head(10))
else:
    print("Item_Name column not found.")

# choose target column
target_col = "Order_Status"

# drop columns you do not want to use
drop_cols = ["Order_ID", "User_ID", "Restaurant_ID", "Driver_ID", "Order_Time", "Item_Name"]
drop_cols = [col for col in drop_cols if col in df.columns]

X = df.drop(columns=drop_cols + [target_col])
y = df[target_col]

# split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# categorical and numeric columns
categorical_cols = X_train.select_dtypes(include=["object", "category"]).columns.tolist()
numeric_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()

# preprocessing
preprocess = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
    ("num", "passthrough", numeric_cols)
])

# model
model = Pipeline([
    ("preprocess", preprocess),
    ("rf", RandomForestClassifier(n_estimators=300, random_state=42))
])

# train
model.fit(X_train, y_train)

# predict
y_pred = model.predict(X_test)


Unique Item_Name: 9
Unique Item_Name_reduced: 7
       Item_Name Item_Name_reduced
0          Sushi             Other
1          Pizza             Pizza
2        Koshary           Koshary
3         Burger            Burger
4          Salad             Other
5       Sandwich          Sandwich
6          Sushi             Other
7  Fried Chicken             Other
8          Pasta             Pasta
9          Pasta             Pasta


C:\Users\joury\AppData\Local\Temp\ipykernel_15040\1465885056.py:51: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X_train.select_dtypes(include=["object", "category"]).columns.tolist()


## Accuracy


In [ ]:
acc = accuracy_score(y_test, y_pred)
print("Accuracy:", acc)

NameError: name 'model' is not defined

## Top Feature Importances

In [ ]:
ohe = model.named_steps["preprocess"].named_transformers_["cat"]
cat_feature_names = ohe.get_feature_names_out(categorical_cols)
all_features = np.concatenate([cat_feature_names, numeric_cols])

importances = model.named_steps["rf"].feature_importances_

fi = pd.DataFrame({
    "Feature": all_features,
    "Importance": importances
}).sort_values(by="Importance", ascending=False)

print("\nTop Feature Importances:")
print(fi.head(10))